# Consolidated Multi-LoRA Serving — Intent + Mode A on One vLLM Server

**Real current state, before this notebook**: two separate Colab GPU sessions, two
separate tunnels.

| Logical model | Deployment today |
|---|---|
| Intent (zero-shot) | raw `MBZUAI-Paris/Nile-Chat-12B`, port 8002, `nile-chat-12b-base`, no adapter (`serve_nilechat12b_base_query_router.ipynb`) |
| Mode A (JSON extraction + reply) | `mennaharmas/raylab-nilechat-12b-v2`, port 8001, **fully merged** model, no LoRA flags (`merge_and_run_nilechat12b_v2.ipynb`) |

Both load the *same base checkpoint*. That's what makes consolidation possible:
vLLM's `--enable-lora` mode runs ONE base-model process and hot-swaps N adapters via
the `"model"` field per request — every adapter must share that one base, which these
already do.

**What this notebook does**: serves the raw base + ONE named LoRA adapter
(`mode-a-lora` = Mode A's real, already-existing adapter repo) on **one** vLLM
process, one tunnel, one GPU — replacing both deployments above. The CQR adapter
(`cqr-lora`) is deliberately not loaded here — see "Client code changes" near the
bottom for how `rewrite_query()` keeps working without it.

**Real, disclosed risk**: switching Mode A from a merged model to an adapter-served
one is a genuine behavior-preserving-*in-theory-only* change — vLLM's LoRA inference
path is not byte-identical to a merged model's forward pass in every implementation
detail. This notebook's own smoke test (below) confirms the server responds correctly
under both model names, but does **not** by itself prove Mode A's accuracy is
unchanged — re-run Mode A's real `val.json` exact-JSON-match evaluation
(`nile_chat_finetune_v2_colab_and_merge.ipynb`'s own Stage-6-shaped cell) against
`"model": "mode-a-lora"` on this consolidated server and compare to the merged
deployment's known baseline before treating this as a drop-in replacement.

Runtime: **Colab, 1× A100 GPU**.

**Before running anything**: `Runtime > Change runtime type > A100 GPU`, then confirm
the GPU is actually attached below.

In [ ]:
!nvidia-smi


If the cell above errors or shows no GPU, stop here and fix the runtime type before
continuing.

### Install

Same proven vLLM/torch/torchvision/torchaudio CUDA-matching fix as every other real
serving notebook in this project (`run_nilechat12b.ipynb`,
`merge_and_run_nilechat12b_v2.ipynb`) — `torchvision` is a real, unguarded vLLM import
requirement; `torchaudio` is deliberately left uninstalled (no matching CUDA build
exists yet, and nothing in this text-only pipeline needs it).

In [ ]:
!pip install -q -U vllm transformers accelerate

import torch

torch_version = torch.__version__.split("+")[0]
torch_cuda = torch.version.cuda
cuda_tag = "cu" + torch_cuda.replace(".", "")

print(f"Detected torch=={torch_version} built for CUDA {torch_cuda} -> installing matched "
      f"torchvision from index {cuda_tag}, leaving torchaudio uninstalled")

!pip install -q "torch=={torch_version}" torchvision --index-url https://download.pytorch.org/whl/{cuda_tag}
!pip uninstall -y -q torchaudio

import subprocess
check = subprocess.run(
    ["python", "-c", "import torch, torchvision; "
     "print('torch:', torch.__version__, torch.version.cuda); "
     "print('torchvision:', torchvision.__version__)"],
    capture_output=True, text=True,
)
print(check.stdout.strip())
assert check.returncode == 0, f"torch/torchvision import failing:\n{check.stderr}"
print("torch/torchvision aligned and importable. torchaudio intentionally left uninstalled.")


### Hugging Face token — REQUIRED

`mennaharmas/raylab-nilechat-12b-v2-lora` is a **private** repo (same `private=True`
Mode A's own merged-model repo already uses in production, which this same token
pattern already proves works for a private repo with vLLM). Without `HF_TOKEN` set,
vLLM's `--lora-modules` download of the adapter will fail with a 401/403, not a clear
"auth" error — set this before the serve cell, not after it fails.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('huggingface')
print("HF_TOKEN set")


### Serve — one base checkpoint, one named LoRA adapter

`--max-lora-rank 16` matches Mode A's real trained rank (`lora_rank: 16` in its
training yaml) — must be raised if the adapter is ever retrained at a higher rank.
`--served-model-name nile-chat-12b-base` keeps the base itself directly addressable
under its existing name (standard vLLM behavior with `--enable-lora` active, not a
workaround) — Intent classification needs no code change at all.

In [ ]:
!nohup vllm serve "MBZUAI-Paris/Nile-Chat-12B" \
    --dtype bfloat16 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.85 \
    --port 8001 \
    --served-model-name nile-chat-12b-base \
    --enable-lora \
    --max-lora-rank 16 \
    --lora-modules mode-a-lora=mennaharmas/raylab-nilechat-12b-v2-lora \
    > vllm.log 2>&1 &


In [ ]:
import time

ready = False
for attempt in range(60):  # up to 10 minutes -- loading a 12B base + 1 adapter
    time.sleep(10)
    log = open("vllm.log").read() if __import__("os").path.exists("vllm.log") else ""
    if "Uvicorn running" in log or "Application startup complete" in log:
        ready = True
        break
    if "Traceback" in log and "ERROR" in log:
        print("vLLM logged an error while loading -- check the tail below.")
        break
    print(f"[{(attempt + 1) * 10}s] still loading...")

!tail -n 80 vllm.log
print("\n--- server ready:", ready, "---\n")


If the tail above didn't show the server ready, stop and fix it before opening a
tunnel -- a tunnel just exposes whatever's on :8001, broken or not.

In [ ]:
import os, re, time

if not os.path.exists("cloudflared-linux-amd64"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

assert os.path.exists("cloudflared-linux-amd64") and os.path.getsize("cloudflared-linux-amd64") > 0, \
    "cloudflared download failed -- re-run this cell, or check Colab's network connectivity"

!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8001 > cloudflared.log 2>&1 &

tunnel_url = None
for _ in range(30):
    time.sleep(2)
    log = open("cloudflared.log").read() if os.path.exists("cloudflared.log") else ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
    if match:
        tunnel_url = match.group(0)
        break

assert tunnel_url, "Tunnel URL not found after 60s -- check cloudflared.log for errors and re-run this cell"
print(f"Tunnel URL: {tunnel_url}")
print("\nDo NOT update your local .env yet -- run the smoke test below first.")


## Smoke test — both logical models, one server

Confirms the adapter hot-swap actually works end to end before you touch any local
config. Both requests hit the SAME `localhost:8001` process, differing only in
the `"model"` field.

In [ ]:
import requests

def smoke_test(model_name, message):
    resp = requests.post(
        "http://localhost:8001/v1/chat/completions",
        json={"model": model_name, "messages": [{"role": "user", "content": message}], "max_tokens": 32},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

print("--- nile-chat-12b-base (Intent, zero-shot) ---")
print(smoke_test("nile-chat-12b-base", "إزيك؟"))

print("\n--- mode-a-lora (Mode A JSON extraction + reply) ---")
print(smoke_test("mode-a-lora", "إزيك؟"))

print("\nBoth logical models responded from the SAME consolidated server -- "
      "adapter hot-swap confirmed working, before any real .env cutover.")


## Before cutting over: re-validate Mode A's real accuracy

The smoke test above only proves the server *responds* under each model name — it
does not prove `mode-a-lora`'s accuracy matches the currently-deployed merged model.
Re-run Mode A's own real `val.json` exact-JSON-match evaluation cell (from
`nile_chat_finetune_v2_colab_and_merge.ipynb`) against `"model": "mode-a-lora"` on
`http://localhost:8001` (this same running server) and compare the resulting accuracy
to the merged deployment's own known baseline. Only cut over once they match — this
is the one real, unproven assumption in the whole consolidation plan.

## Client code changes

**Already applied to the local codebase** (safe, additive, zero behavior change to
the currently-running system — verified: `QUERY_ROUTER_CQR_MODEL_NAME` defaults to
`None`, so `rewrite_query()` keeps using the exact same model `classify_intent()`
does until this is explicitly set):

- `src/stores/query_router/providers/NileChat12BBaseProvider.py` — constructor gained
  `cqr_model_name: str | None = None`; `_post_chat_completion` gained an optional
  `model` override param; `rewrite_query()` now passes `self.cqr_model_name`
  (`classify_intent()` is unchanged, still uses `self.model_name`).
- `src/stores/query_router/QueryRouterProviderFactory.py` — threads
  `config.QUERY_ROUTER_CQR_MODEL_NAME` into the provider's `cqr_model_name` param.
- `src/helpers/config.py` / `src/.env.example` — new optional setting,
  `QUERY_ROUTER_CQR_MODEL_NAME: str | None = None`.

**NOT applied automatically — do this manually, only after the val.json re-validation
above passes.** This notebook runs on a remote Colab VM and has no access to your
local Windows machine's filesystem, so this is a real edit you make yourself in
`src/.env` (this is the actual live cutover — the one step in this whole plan with
real production blast radius):

```
GENERATION_BASE_URL=<the tunnel URL printed above>
QUERY_ROUTER_BASE_URL=<the SAME tunnel URL>
GENERATION_MODEL_NAME=mode-a-lora
QUERY_ROUTER_MODEL_NAME=nile-chat-12b-base   # unchanged
```

Then restart (or let `--reload` pick up) the API process. `classify_intent()` keeps
using `nile-chat-12b-base` exactly as it always has; `rewrite_query()` also keeps
using `nile-chat-12b-base` (no `QUERY_ROUTER_CQR_MODEL_NAME` is set, so it falls back
to the same model `classify_intent()` uses — see the note at the top of this cell);
only Mode A generation routes to its own dedicated adapter on the one consolidated
server.